This notebook profiles the email leads table over the last twelve months. Throughout, a usable lead means a non spam lead, defined by IS_SPAM_FINAL equal to zero, in the trailing twelve month window.

In [ ]:
%%sql -r dataframe_2
SELECT
  COUNT(*)                                                             AS leads_all_time,
  COUNT_IF(DAY >= DATEADD(month,-12,CURRENT_DATE))                     AS leads_last_12mo_incl_spam,
  COUNT_IF(DAY >= DATEADD(month,-12,CURRENT_DATE) AND IS_SPAM_FINAL=1) AS spam_last_12mo,
  COUNT_IF(DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0) AS usable_last_12mo,
  ROUND(COUNT_IF(DAY >= DATEADD(month,-12,CURRENT_DATE) AND IS_SPAM_FINAL=1)
        *100.0/NULLIF(COUNT_IF(DAY >= DATEADD(month,-12,CURRENT_DATE)),0),1) AS spam_pct_last_12mo
FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS;

This measures how many distinct buyers those usable leads come from, the average number of leads per buyer, and how many duplicate lead identifiers exist. 

In [ ]:
%%sql -r dataframe_4
WITH base AS (
  SELECT LEAD_ID, SENDER_EMAIL
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
)
SELECT
  COUNT(*)                                                     AS usable_leads,
  COUNT(DISTINCT SENDER_EMAIL)                                 AS distinct_buyers,
  ROUND(COUNT(*)*1.0/NULLIF(COUNT(DISTINCT SENDER_EMAIL),0),2) AS avg_leads_per_buyer,
  COUNT(*) - COUNT(DISTINCT LEAD_ID)                           AS duplicate_lead_id_rows
FROM base;

This is the monthly lead volume that drives the seasonality view. Volume peaks in spring and troughs in winter, swinging roughly two and a half to three times across the year.

In [ ]:
%%sql -r dataframe_5
SELECT TO_CHAR(DATE_TRUNC('month', DAY),'YYYY-MM') AS month, COUNT(*) AS leads
FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
GROUP BY 1 ORDER BY 1;

This is the field completeness scan. Each number is the share of usable leads where the field is filled in. It shows which candidate features are well populated and which are sparse.


In [ ]:
%%sql -r dataframe_6
WITH base AS (
  SELECT * FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
), n AS (SELECT COUNT(*) c FROM base)
SELECT
  (SELECT c FROM n) AS total_usable,
  ROUND(COUNT_IF(NULLIF(TRIM(CUSTOMER_BT_MEMBERSHIP_TIER),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS bt_membership_tier,
  ROUND(COUNT_IF(NULLIF(TRIM(LAST_TOUCH_CHANNEL),'')        IS NOT NULL)*100.0/(SELECT c FROM n),1) AS last_touch_channel,
  ROUND(COUNT_IF(NULLIF(TRIM(CUSTOMER_COMMENTS),'')         IS NOT NULL)*100.0/(SELECT c FROM n),1) AS customer_comments,
  ROUND(COUNT_IF(NULLIF(TRIM(CUSTOMER_TYPE),'')             IS NOT NULL)*100.0/(SELECT c FROM n),1) AS customer_type,
  ROUND(COUNT_IF(NULLIF(TRIM(SENDER_EMAIL_DOMAIN),'')       IS NOT NULL)*100.0/(SELECT c FROM n),1) AS sender_email_domain,
  ROUND(COUNT_IF(NULLIF(TRIM(PRODUCT_CLASS),'')   IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_class,
  ROUND(COUNT_IF(NULLIF(TRIM(PRODUCT_MAKE),'')    IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_make,
  ROUND(COUNT_IF(PRODUCT_YEAR   IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_year,
  ROUND(COUNT_IF(PRODUCT_LENGTH IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_length,
  ROUND(COUNT_IF(NULLIF(TRIM(PRODUCT_CONDITION),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_condition,
  ROUND(COUNT_IF(PRODUCT_PRICE  IS NOT NULL)*100.0/(SELECT c FROM n),1) AS product_price,
  ROUND(COUNT_IF(NULLIF(TRIM(PHONE_NUMBER),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS phone_in_lead,
  ROUND(COUNT_IF(NULLIF(TRIM(GA_CLIENT_ID),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS ga_client_id,
  ROUND(COUNT_IF(NULLIF(TRIM(NEW_VS_RETURNED),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS new_vs_returned,
  ROUND(COUNT_IF(USER_FRAUD_SCORE IS NOT NULL)*100.0/(SELECT c FROM n),1) AS fraud_score,
  ROUND(COUNT_IF(USER_LISTING_DISTANCE_IN_MILES IS NOT NULL)*100.0/(SELECT c FROM n),1) AS buyer_listing_distance,
  ROUND(COUNT_IF(USER_PARTY_ID IS NOT NULL)*100.0/(SELECT c FROM n),1) AS buyer_account_id,
  ROUND(COUNT_IF(CAPTCHA_SCORE IS NOT NULL)*100.0/(SELECT c FROM n),1) AS captcha_score,
  ROUND(COUNT_IF(NULLIF(TRIM(EMAIL_LEAD_SUBJECT),'') IS NOT NULL)*100.0/(SELECT c FROM n),1) AS email_subject
FROM base;